# 🚀 DeepNT+: Hybrid Path-Centric Graph Neural Network for Robust & Dynamic Network Tomography
### Complete Master Google Colab Benchmark Notebook
**Thesis Project:** *Hybrid Path-Centric GNN Approach for Robust and Dynamic Network Tomography (DeepNT+)*  
**Author:** Atik Morshed (ID: 210153) | **Department:** Computer Science & Engineering, JUST  
**Features Included:**
1. **Laplace Negative Log-Likelihood (NLL) Loss** (Outlier-resilient latency regression)
2. **Coverage-Aware Residual Gate** ($b_g = -2.0$, anchoring on convex tomography to prevent degradation)
3. **Dynamic Context-Rate Warm Start** (eliminating IPv6 prior mismatch)
4. **CosineAnnealingLR Scheduler** (smooth convergence to flat minima)
5. **Monitor Baseline Offset Normalization** (zero-leakage vantage point transfer)
6. **5-Seed Comprehensive Benchmarks** (`seeds = 0, 1, 2, 3, 4`) across IPv4 and IPv6
7. **Leave-Monitor-Out (LMO)** Cross-Vantage Generalization Test (`dac-bd` holdout)
8. **Empirical Active Learning Probing Curves** (15%, 30%, 45% probing fractions comparing BALD vs. Random vs. Edge Coverage)
9. **Publication-Ready Figures & Calibration Plots**


In [ ]:
# ==============================================================================
# Cell 1: Environment Setup & Google Drive Mount
# ==============================================================================
import os
import sys
import time
import math
import copy
from pathlib import Path

# Detect environment (Google Colab vs. Local Machine)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    print(" Running in Google Colab environment. Drive mounted successfully.")
except ImportError:
    IN_COLAB = False
    print(" Running in Local Jupyter / Workspace environment.")

# Set Project Roots
if IN_COLAB:
    BASE_DIR = Path("/content/drive/My Drive/Thesis")
else:
    BASE_DIR = Path("C:/Users/KINGDOM/Desktop/Thesis")

COLAB_DIR = BASE_DIR / "colab"
OUTPUT_DIR = BASE_DIR / "DeepNT_Output"

# Ensure colab directory is on Python path
if COLAB_DIR.exists():
    sys.path.insert(0, str(COLAB_DIR))
elif Path.cwd().exists():
    sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

print(f" PyTorch Version: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")
print(f" Working Directory: {Path.cwd()}")


In [ ]:
# ==============================================================================
# Cell 2: Benchmark Configuration & Dataset Selector
# ==============================================================================
# Select which dataset to evaluate:
#   'run_03'   : Compressed IPv4 5-graph dataset (Recommended for fast, dense benchmarks)
#   'run_02'   : Balanced IPv4 15-graph dataset (~400 nodes/graph)
#   'run_ipv6' : Global CAIDA Ark IPv6 5-graph dataset (Massive global topology)
DATASET_CHOICE = 'run_03'  # @param ['run_03', 'run_02', 'run_ipv6']

DATASET_PATHS = {
    'run_02': OUTPUT_DIR / 'data' / 'processed_15graphs_backup',
    'run_03': OUTPUT_DIR / 'data' / 'processed',
    'run_ipv6': OUTPUT_DIR / 'reconstructed_dataset' / 'ipv6',
}

DATASET_PATH = DATASET_PATHS[DATASET_CHOICE]
OUT_EXPERIMENT_DIR = OUTPUT_DIR / "experiments" / "deepnt" / "colab_benchmark" / DATASET_CHOICE
OUT_EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

# Experimental Hyperparameters
SEEDS = [0, 1, 2, 3, 4]  # 5-seed protocol for robust academic reporting
EPOCHS = 40              # 40 epochs is ideal with CosineAnnealingLR & patience 15
BATCH_PATHS = 128
LR = 1e-3
WEIGHT_DECAY = 1e-4
LOSS_TYPE = 'laplace'     # 'laplace' (Robust L1) vs 'gaussian'
USE_COSINE_LR = True     # Cosine Annealing learning rate schedule
RESIDUAL_REG = 0.05      # Penalty weight to anchor neural refinement on convex solution

print(f" Selected Dataset: {DATASET_CHOICE}")
print(f" Dataset Path    : {DATASET_PATH}")
print(f" Output Directory: {OUT_EXPERIMENT_DIR}")
print(f" Seeds: {SEEDS} | Epochs: {EPOCHS} | Loss: {LOSS_TYPE} | Cosine LR: {USE_COSINE_LR}")


In [ ]:
# ==============================================================================
# Cell 3: Dataset Loading & Topology Inspection
# ==============================================================================
from deepnt_plus.data import load_dataset, describe, NODE_FEATURES

print(f"Loading dataset from: {DATASET_PATH} ...")
graphs, norm = load_dataset(DATASET_PATH)
overview = describe(graphs)

print("
=== Dataset Graph Overview ===")
print(overview.to_string(index=False))

total_paths = overview['paths'].sum()
total_obs = overview['observed'].sum()
total_hidden = overview['hidden'].sum()
print(f"
 Total Paths: {total_paths:,} | Observed Context: {total_obs:,} ({total_obs/total_paths*100:.1f}%) | Hidden Targets: {total_hidden:,}")
print(f" Log-RTT Mean: {norm['log_rtt_mean']:.4f} | Std: {norm['log_rtt_std']:.4f} | Prior Edge Delay: {norm['prior_edge_ms']:.2f} ms")


## 🧠 DeepNT+ Architecture & Improved Residual Fusion
DeepNT+ integrates 5 state-of-the-art inductive biases:
1. **Split-Weight Spatial GNN:** Explicitly treats forward and reverse routing links separately.
2. **Additive Tomography Layer:** Enforces non-negative link latencies $x_e \ge 0$ such that path latency strictly equals $A x + b$.
3. **Coverage-Aware Residual Gate:**
   $$\hat{y} = \hat{y}_{	ext{convex}} + g \cdot (\hat{y}_{	ext{gnn}} - \hat{y}_{	ext{convex}})$$
   Initialized with gate bias $b_g = -2.0$ ($g pprox 0.12$), anchoring on the classical convex solution to guarantee zero degradation.
4. **Dynamic Context-Rate Warm-Start:** Initializes link priors $x_0 = 	ext{median}(y_{	ext{ctx}} / h_{	ext{ctx}})$, eliminating prior scale mismatch on unseen vantage points.
5. **Heteroscedastic Gaussian Uncertainty Head:** Quantifies aleatoric and epistemic uncertainties via Monte Carlo Dropout.


In [ ]:
# ==============================================================================
# Cell 4: Model Architecture & Training Function Definition
# ==============================================================================
from deepnt_plus.model import DeepNTPlus, ModelConfig
from deepnt_plus.train import TrainConfig, train_model, evaluate, score
from deepnt_plus.baselines import run_structural_baselines, run_convex_baseline

NEURAL_MODELS = {
    'DeepNT+ (full, improved)': dict(gnn_kind='split', tomography=True, convex=True, temporal=True, uncertainty=True),
    'DeepNT (static, causal)': dict(gnn_kind='split', tomography=True, convex=True, temporal=False, uncertainty=True),
    'GAT': dict(gnn_kind='gat', tomography=False, convex=False, temporal=False, uncertainty=False),
    'GraphSAGE': dict(gnn_kind='sage', tomography=False, convex=False, temporal=False, uncertainty=False),
}

print(" Models configured:")
for m, cfg in NEURAL_MODELS.items():
    print(f"  • {m:26s}: {cfg}")


In [ ]:
# ==============================================================================
# Cell 5: Run 5-Seed Clean Benchmark Suite
# ==============================================================================
all_preds = []
pred_csv = OUT_EXPERIMENT_DIR / 'predictions_all.csv'

# 1. Evaluate Classical Baselines across seeds
print("
--- Running Classical Baselines (Median, Hop-count, Warm-Start Convex, RF, MLP) ---")
for seed in SEEDS:
    struct_df = run_structural_baselines(graphs, seed=seed)
    struct_df['seed'] = seed
    all_preds.append(struct_df)

convex_df = run_convex_baseline(graphs, norm, warm_start=True)
for seed in SEEDS:
    c_df = convex_df.copy()
    c_df['seed'] = seed
    all_preds.append(c_df)

# 2. Evaluate Neural Models across 5 Seeds
for mname, mparams in NEURAL_MODELS.items():
    print(f"
==========================================")
    print(f" Training: {mname}")
    print(f"==========================================")
    for seed in SEEDS:
        t0 = time.time()
        mcfg = ModelConfig(
            node_dim=len(NODE_FEATURES),
            has_loss=False,
            **mparams
        )
        tcfg = TrainConfig(
            epochs=EPOCHS,
            patience=15,
            loss_type=LOSS_TYPE,
            use_cosine_lr=USE_COSINE_LR,
            residual_reg_weight=RESIDUAL_REG,
        )

        torch.manual_seed(seed)
        np.random.seed(seed)
        model = DeepNTPlus(mcfg, norm)

        hist = train_model(model, graphs, tcfg, seed=seed)
        frames = []
        for split in ('val', 'test'):
            f = evaluate(model, graphs, split, tcfg, mc=10 if mcfg.uncertainty else 0)
            f['model'] = mname
            f['seed'] = seed
            frames.append(f)

        res_df = pd.concat(frames, ignore_index=True)
        all_preds.append(res_df)

        test_score = score(res_df[res_df['split'] == 'test'])
        val_score = score(res_df[res_df['split'] == 'val'])
        print(f"  Seed {seed} ({time.time()-t0:.1f}s) | Val MAE: {val_score['MAE']:.2f} ms | Test MAE: {test_score['MAE']:.2f} ms | Test RMSE: {test_score['RMSE']:.2f} ms | R2: {test_score['R2']:.3f}")

# Save all predictions
master_predictions = pd.concat(all_preds, ignore_index=True)
master_predictions.to_csv(pred_csv, index=False)
print(f"
 All predictions successfully saved to: {pred_csv}")


In [ ]:
# ==============================================================================
# Cell 6: Compile & Display 5-Seed Results Summary Table
# ==============================================================================
test_preds = master_predictions[master_predictions['split'] == 'test']

rows = []
for (m, sd), grp in test_preds.groupby(['model', 'seed']):
    sc = score(grp)
    sc.update({'model': m, 'seed': sd})
    rows.append(sc)

per_seed_metrics = pd.DataFrame(rows)
per_seed_metrics.to_csv(OUT_EXPERIMENT_DIR / 'metrics_per_seed.csv', index=False)

num_cols = [c for c in ['MAE', 'RMSE', 'R2', 'coverage_90', 'ECE_reg', 'NLL'] if c in per_seed_metrics.columns]
agg_dict = {c: ['mean', 'std'] for c in num_cols}
summary_table = per_seed_metrics.groupby('model').agg(agg_dict).reset_index()
summary_table.columns = ['_'.join(c).strip('_') for c in summary_table.columns]
summary_table = summary_table.sort_values('MAE_mean').reset_index(drop=True)

summary_table.to_csv(OUT_EXPERIMENT_DIR / 'summary_metrics_5seeds.csv', index=False)

print(f"
==========================================================================================")
print(f"                      DEEPNT+ 5-SEED MASTER BENCHMARK RESULTS ({DATASET_CHOICE.upper()})")
print(f"==========================================================================================")
for _, r in summary_table.iterrows():
    cov_str = f" | Cov90: {r['coverage_90_mean']:.3f} ± {r['coverage_90_std']:.3f}" if 'coverage_90_mean' in r and pd.notna(r['coverage_90_mean']) else ""
    ece_str = f" | ECE: {r['ECE_reg_mean']:.3f} ± {r['ECE_reg_std']:.3f}" if 'ECE_reg_mean' in r and pd.notna(r['ECE_reg_mean']) else ""
    print(f" {r['model']:30s} | MAE: {r['MAE_mean']:6.2f} ± {r['MAE_std']:4.2f} ms | RMSE: {r['RMSE_mean']:6.2f} ± {r['RMSE_std']:4.2f} ms | R2: {r['R2_mean']:6.3f} ± {r['R2_std']:5.3f}{cov_str}{ece_str}")
print(f"==========================================================================================")


In [ ]:
# ==============================================================================
# Cell 7: Leave-Monitor-Out (LMO) Cross-Vantage Generalization Test
# ==============================================================================
# Evaluates transferability when an entire monitor is held out from training
if DATASET_CHOICE in ['run_02', 'run_03']:
    print("
--- Running Leave-Monitor-Out Test (dac-bd held out) ---")
    from deepnt_plus.data import load_dataset
    
    lmo_graphs, lmo_norm = load_dataset(DATASET_PATH, holdout_monitor='dac-bd')
    
    lmo_models = {
        'Convex tomography (classical)': None,
        'DeepNT (static, causal)': dict(temporal=False),
        'DeepNT+ (full, improved)': dict(temporal=True),
        'GAT': dict(gnn_kind='gat', tomography=False, convex=False, temporal=False, uncertainty=False),
        'Median': None,
        'Hop-count linear': None,
        'Random forest': None,
    }
    
    lmo_records = []
    # Classical baselines
    s_lmo = run_structural_baselines(lmo_graphs, seed=0)
    c_lmo = run_convex_baseline(lmo_graphs, lmo_norm, warm_start=True)
    all_lmo = pd.concat([s_lmo, c_lmo], ignore_index=True)
    
    for mname, mparams in lmo_models.items():
        if mparams is not None:
            mcfg = ModelConfig(node_dim=len(NODE_FEATURES), **mparams)
            tcfg = TrainConfig(epochs=EPOCHS, patience=15, loss_type=LOSS_TYPE, use_cosine_lr=USE_COSINE_LR, residual_reg_weight=RESIDUAL_REG)
            torch.manual_seed(0)
            model = DeepNTPlus(mcfg, lmo_norm)
            train_model(model, lmo_graphs, tcfg, seed=0)
            pred = evaluate(model, lmo_graphs, 'test', tcfg, mc=10 if mcfg.uncertainty else 0)
            pred['model'] = mname
            all_lmo = pd.concat([all_lmo, pred], ignore_index=True)
            
    test_lmo = all_lmo[all_lmo['split'] == 'test']
    lmo_summary = []
    for m in test_lmo['model'].unique():
        sc = score(test_lmo[test_lmo['model'] == m])
        sc['model'] = m
        lmo_summary.append(sc)
        
    lmo_df = pd.DataFrame(lmo_summary).sort_values('MAE').reset_index(drop=True)
    print("
=== Leave-Monitor-Out (dac-bd) Test Results ===")
    for _, r in lmo_df.iterrows():
        print(f"  {r['model']:30s} | MAE: {r['MAE']:6.2f} ms | RMSE: {r['RMSE']:6.2f} ms | R2: {r['R2']:6.3f}")
else:
    print(" LMO is pre-configured on IPv4 topologies with heterogeneous global monitors.")


In [ ]:
# ==============================================================================
# Cell 8: Empirical Active Learning Probing Curves (Objective 4)
# ==============================================================================
from deepnt_plus.train import eval_episode, predict_frame
from deepnt_plus import active

print("
--- Simulating Active Path Probing Across Budgets (15%, 30%, 45%) ---")
# Train static model on seed 0 for acquisition
mcfg = ModelConfig(node_dim=len(NODE_FEATURES), temporal=False, uncertainty=True)
tcfg = TrainConfig(epochs=EPOCHS, patience=15, loss_type=LOSS_TYPE, use_cosine_lr=USE_COSINE_LR)
torch.manual_seed(0)
al_model = DeepNTPlus(mcfg, norm)
train_model(al_model, graphs, tcfg, seed=0)

test_graphs = [g for g in graphs.values() if g.split == 'test']
fractions = [0.15, 0.30, 0.45]
al_rows = []

for frac in fractions:
    for strategy in ('bald', 'random', 'edge_coverage'):
        strats_seeds = range(3) if strategy == 'random' else [0]
        maes, rmses = [], []
        for s_seed in strats_seeds:
            rng = np.random.default_rng(s_seed)
            for g in test_graphs:
                ctx0, hidden = eval_episode(g)
                all_paths = ctx0.tolist() + hidden.tolist()
                target_k = int(round(frac * len(all_paths)))
                init_k = max(2, int(0.05 * len(all_paths)))
                order = rng.permutation(all_paths)
                curr_ctx = order[:init_k].tolist()
                pool = order[init_k:].tolist()
                
                while len(curr_ctx) < target_k and len(pool) > 0:
                    budget_left = target_k - len(curr_ctx)
                    step_k = min(budget_left, max(1, int(0.05 * len(all_paths))))
                    use_mc = 15 if strategy == 'bald' else 0
                    frame = predict_frame(al_model, g, tcfg, ctx=torch.tensor(curr_ctx), tgt=torch.tensor(pool), mc=use_mc)
                    scores = active._scores(strategy, frame, g, curr_ctx, pool, rng)
                    pick_idx = np.argsort(-scores)[:step_k]
                    curr_ctx += [pool[i] for i in pick_idx]
                    pool = [p for i, p in enumerate(pool) if i not in set(pick_idx.tolist())]
                    
                eval_tgt = [p for p in hidden.tolist() if p not in set(curr_ctx)] or pool
                frame_eval = predict_frame(al_model, g, tcfg, ctx=torch.tensor(curr_ctx), tgt=torch.tensor(eval_tgt), mc=0)
                err = np.abs(frame_eval['y_pred'] - frame_eval['y_true'])
                maes.append(float(err.mean()))
                rmses.append(float(np.sqrt((err**2).mean())))
                
        al_rows.append({
            'budget_fraction': frac,
            'budget_pct': f"{int(frac*100)}%",
            'strategy': strategy,
            'MAE_mean': float(np.mean(maes)),
            'MAE_std': float(np.std(maes)),
            'RMSE_mean': float(np.mean(rmses)),
            'RMSE_std': float(np.std(rmses))
        })
        print(f"  Budget: {int(frac*100)}% | Strategy: {strategy:15s} | Test MAE: {np.mean(maes):.2f} ± {np.std(maes):.2f} ms")

al_df = pd.DataFrame(al_rows)
al_df.to_csv(OUT_EXPERIMENT_DIR / 'active_learning_probing_curves.csv', index=False)


In [ ]:
# ==============================================================================
# Cell 9: Publication-Quality Visualizations
# ==============================================================================
from deepnt_plus.metrics import reliability_curve

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Model Comparison Bar Chart (MAE with Std)
models_to_plot = summary_table.sort_values('MAE_mean', ascending=False)
y_pos = np.arange(len(models_to_plot))
axes[0].barh(y_pos, models_to_plot['MAE_mean'], xerr=models_to_plot['MAE_std'].fillna(0),
             color='tab:blue', alpha=0.8, capsize=4)
axes[0].set_yticks(y_pos)
axes[0].set_yticklabels(models_to_plot['model'])
axes[0].set_xlabel('Test MAE (ms)')
axes[0].set_title(f'Model Comparison: 5-Seed Test MAE ({DATASET_CHOICE.upper()})')
axes[0].grid(axis='x', linestyle='--', alpha=0.6)

# Plot 2: Delay Scatter Plot (DeepNT+ full)
d_full = test_preds[test_preds['model'] == 'DeepNT+ (full, improved)']
if not d_full.empty:
    lim = [0, max(d_full['y_true'].quantile(0.99), d_full['y_pred'].quantile(0.99)) * 1.1]
    axes[1].scatter(d_full['y_true'], d_full['y_pred'], alpha=0.5, s=15, color='tab:green', label='Predicted Paths')
    axes[1].plot(lim, lim, 'k--', lw=1.5, label='Ideal y=x')
    axes[1].set_xlim(lim); axes[1].set_ylim(lim)
    axes[1].set_xlabel('True RTT (ms)')
    axes[1].set_ylabel('Predicted RTT (ms)')
    axes[1].set_title('DeepNT+ Delay Predictions vs. Ground Truth')
    axes[1].legend()
    axes[1].grid(True, linestyle='--', alpha=0.5)

# Plot 3: Active Learning Probing Curve
if not al_df.empty:
    for strat, c in [('bald', 'tab:purple'), ('edge_coverage', 'tab:red'), ('random', 'tab:gray')]:
        sub = al_df[al_df['strategy'] == strat]
        axes[2].plot(sub['budget_fraction'] * 100, sub['MAE_mean'], marker='o', label=strat.replace('_', ' ').title(), color=c, lw=2)
        axes[2].fill_between(sub['budget_fraction'] * 100, sub['MAE_mean'] - sub['MAE_std'], sub['MAE_mean'] + sub['MAE_std'], color=c, alpha=0.15)
    axes[2].set_xlabel('Probing Budget (% Context Paths)')
    axes[2].set_ylabel('Test MAE (ms)')
    axes[2].set_title('Active Probing: MAE vs. Measurement Budget')
    axes[2].legend()
    axes[2].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
fig_path = OUT_EXPERIMENT_DIR / 'deepnt_plus_master_figures.png'
plt.savefig(fig_path, dpi=200)
plt.show()
print(f" Figures saved to: {fig_path}")


In [ ]:
# ==============================================================================
# Cell 10: Final Export Summary & Drive Persistence
# ==============================================================================
print(f"
==========================================================================================")
print(f" ALL BENCHMARKS & FIGURES SUCCESSFULLY COMPUTED AND PERSISTED!")
print(f" Output Location: {OUT_EXPERIMENT_DIR}")
print(f" Files Created:")
for p in sorted(OUT_EXPERIMENT_DIR.glob('*')):
    if p.is_file():
        size_kb = p.stat().st_size / 1024
        print(f"   • {p.name:35s} ({size_kb:.1f} KB)")
print(f"==========================================================================================")
